# Lesson 15 Lab — FP8 KV Cache: Capacity and Fidelity

**Puzzle:** Does halving KV element width double safe long-context concurrency without changing answers?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

KV quantization targets state that grows with context rather than model weights. It can materially extend capacity, but scale calibration, numerical drift, kernel support, and non-KV reserves prevent a free two-times service claim.


## 0. Predict before running

1. Predict whether the RTX 5090 build accepts FP8 KV.
2. Compare theoretical bytes per token.
3. Choose rollback evidence for a long-context route.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The native experiment runs the same deterministic prompts with automatic and FP8 KV cache dtypes, records success or the exact failure, compares token IDs, and pairs the result with the first-order byte ratio.

- KV dtype is independent of weight dtype.
- Theoretical KV bytes can halve while total VRAM falls by less.
- Token equality on a small suite is a regression check, not a quality proof.


## 2. Derive the mechanism

FP8 stores each cached key/value element in one byte rather than BF16's two, plus scale metadata. Attention must dequantize or consume that representation through a supported path. Static or dynamic scales determine range and error. Even perfect twofold KV compression does not halve weight or workspace memory.

### Mechanism at a glance

```mermaid
flowchart LR
  K["BF16 KV vectors"] --> Q["scale + FP8 encode"]
  Q --> C["smaller cache blocks"]
  C --> A["attention read/dequantize"]
  A --> O["logits and token regression"]
  C --> M["long-context capacity test"]
```

### Walk it step by step

1. **Separate weight and cache dtypes.** Keep model weights fixed during the comparison.
2. **Account for scales.** Record how FP8 values are calibrated or dynamically scaled.
3. **Test native execution.** Retain initialization, token, latency, and memory evidence.
4. **Sweep long contexts.** Capacity value appears only when KV is a material budget term.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 15
LESSON_TITLE = 'FP8 KV Cache: Capacity and Fidelity'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260827
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | automatic KV dtype |
| Candidate | FP8 KV dtype with the same BF16 weights |
| Held constant | model, prompts, greedy sampling, maximum length, GPU, and engine version |
| Measurements | initialization success, output token equality, elapsed time, and theoretical KV byte ratio |
| Evidence | `native-backend` |

**Experiment:** Run matched native generations with BF16/auto and FP8 KV configurations, retaining success, tokens, and timing.


## 5. Inspect the experiment code

The code destroys the first engine before creating the second and records configuration failures verbatim. It avoids presenting a failed initialization as a performance measurement.

Do not execute until the code matches the frozen table.


In [2]:
prompts=["Explain KV cache quantization briefly.","Name one FP8 calibration risk."]
params=SamplingParams(temperature=0.0,max_tokens=16,seed=SEED)
def run_kv(dtype):
    row={"success":False,"elapsed_s":None,"records":[],"error":None}
    try:
        engine=LLM(**base_engine_args(max_model_len=1024,kv_cache_dtype=dtype)); tick=time.perf_counter()
        outputs=engine.generate(prompts,params,use_tqdm=False)
        row.update(success=True,elapsed_s=time.perf_counter()-tick,records=[output_record(x) for x in outputs])
        del engine; gc.collect(); torch.cuda.empty_cache()
    except Exception as exc:
        row["error"]=f"{type(exc).__name__}: {exc}"; gc.collect(); torch.cuda.empty_cache()
    return row
auto=run_kv("auto"); fp8=run_kv("fp8")
auto_ids=[x["token_ids"] for x in auto["records"]]; fp8_ids=[x["token_ids"] for x in fp8["records"]]
metrics={"auto":auto,"fp8":fp8,"theoretical_kv_capacity_ratio":2.0,
         "token_sequences_equal":bool(auto["success"] and fp8["success"] and auto_ids==fp8_ids)}
analysis=(f"Auto/FP8 success={auto['success']}/{fp8['success']}; leading KV capacity ratio is 2× and "
          f"matched greedy tokens equal={metrics['token_sequences_equal']}. Short prompts do not prove "
          "long-context capacity or task quality.")


INFO 08-13 00:20:45 [api_utils.py:273] non-default args: {'tokenizer': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'bfloat16', 'seed': 20260827, 'max_model_len': 1024, 'gpu_memory_utilization': 0.45, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'model': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct'}


INFO 08-13 00:20:45 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 08-13 00:20:45 [model.py:1883] Using max model len 1024


INFO 08-13 00:20:45 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.


WARNING 08-13 00:20:45 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


...

(EngineCore pid=651862) INFO 08-13 00:21:15 [gpu_worker.py:563] Available KV cache memory: 10.36 GiB
(EngineCore pid=651862) INFO 08-13 00:21:15 [kv_cache_utils.py:2235] GPU KV cache size: 776,176 tokens
(EngineCore pid=651862) INFO 08-13 00:21:15 [kv_cache_utils.py:2236] Maximum concurrency for 1,024 tokens per request: 757.98x
(EngineCore pid=651862) INFO 08-13 00:21:15 [flashinfer.py:824] FlashInfer resolved query dtypes: prefill=torch.bfloat16, decode=torch.bfloat16, decode_backend=flashinfer-native, kv_cache_dtype=torch.float8_e4m3fn, arch=sm120


(EngineCore pid=651862) INFO 08-13 00:21:15 [kernel_warmup.py:256] Using FlashInfer autotune cache file: <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/eb105e4d09b8e78088bd42168bc114273db1d41da9bf61935c24b4bf8b5acd5a/autotune_configs.json
(EngineCore pid=651862) ERROR 08-13 00:21:16 [core.py:1349] EngineCore failed to start.
(EngineCore pid=651862) ERROR 08-13 00:21:16 [core.py:1349] Traceback (most recent call last):
(EngineCore pid=651862) ERROR 08-13 00:21:16 [core.py:1349]   File "<remote-home>/vllm-ch03/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1318, in run_engine_core
(EngineCore pid=651862) ERROR 08-13 00:21:16 [core.py:1349]     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=651862) ERROR 08-13 00:21:16 [core.py:1349]                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=651862) ERROR 08-13 00:21:16 [core.py:1349]   File "<remote-home>/vllm-ch03/lib/python3.12/si

(EngineCore pid=651862) 2026-08-13 00:21:15,952 - INFO - autotuner.py:829 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=651862) 2026-08-13 00:21:16,033 - INFO - autotuner.py:852 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=651862) Process EngineCore:
(EngineCore pid=651862) Traceback (most recent call last):
(EngineCore pid=651862)   File "<remote-home>/miniconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=651862)     self.run()
(EngineCore pid=651862)   File "<remote-home>/miniconda3/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=651862)     self._target(*self._args, **self._kwargs)
(EngineCore pid=651862)   File "<remote-home>/vllm-ch03/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 1353, in run_engine_core
(EngineCore pid=651862)     raise e
(EngineCore pid=651862)   File "<remote-home>/vllm-ch03/lib/python3.12/site-packages/vllm/v1/engine/core.

[rank0]:[W813 00:21:16.470538285 ProcessGroupNCCL.cpp:1624] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


INFO 08-13 00:21:17 [utils.py:612] [shutdown] Process manager: send sigterm to process EngineCore


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Auto success | yes |
| FP8 success | no |
| Theoretical KV ratio | 2.000x |
| Token sequences equal | no |
| Auto elapsed | 0.182354 |
| FP8 elapsed | not measured |


## 7. Explain the result

Auto/FP8 success=True/False; leading KV capacity ratio is 2× and matched greedy tokens equal=False. Short prompts do not prove long-context capacity or task quality.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 15, "title": 'FP8 KV Cache: Capacity and Fidelity', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": "FP8 can halve the leading KV payload; the native A/B establishes only this model/build's execution and small-suite token behavior.",
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 15,
  "title": "FP8 KV Cache: Capacity and Fidelity",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260827
  },
  "evidence_label": "native-backend",
  "metrics": {
    "auto": {
      "success": true,
      "elapsed_s": 0.18235449399799109,
      "records": [
        {
          "request_id": "0",
          "prompt_tokens": 8,
          "output_tokens": 16,
          "token_ids": [
            84648,
            6500,
            10272,
            2022,
            374,
            264,
            14762,
            1483,
            304,
            5662,
            6832,
            311,
            7949,
            279,
            4938,
            42872
          ],
          "text_preview": " KV cache quantization is a technique used in machine learning to red

## 9. Make the bounded decision

> FP8 can halve the leading KV payload; the native A/B establishes only this model/build's execution and small-suite token behavior.

**Acceptance/rollback:** Adopt FP8 KV only when native long-context capacity rises, task slices pass, and TTFT/ITL do not violate gates.

**Failure analysis:** Short prompts barely exercise cache capacity. Missing calibration scales or a different backend can alter accuracy and speed, while allocator reserves prevent exact 2× concurrency.


## 10. Extend the evidence

Calibrate scales on representative contexts, sweep lengths to the admission limit, compare output distributions, and inspect cache-block capacity plus engine metrics.

The full boundary and references are in [`README.md`](README.md).
